In [1]:
from schemas.event import Event, EventClass
from annotation.tools.event_classifier import hierarchical_event_classification, extract_event_name, extract_event_agents
from utils.loader import load_csv, load_json_folder, save_structured_folktale
from annotation.tools.place_extractor import extract_places
from annotation.tools.agent_extractor import extract_agents
from annotation.tools.genre_extractor import extract_genre
from annotation.tools.event_chunking import extract_story_segments
from annotation.tools.event_extractor import extract_event_elements
from annotation.tools.object_extractor import extract_objects
from annotation.tools.relationship_extractor import extract_relationships
from utils.regex_utils import clean_regex, title_case_to_snake_case
from utils.models import get_llm
from schemas.folktale import Folktale
from pandas import DataFrame
from loguru import logger
from tqdm import tqdm
import re
import os
import json

In [2]:
from schemas.folktale import Genre, GenreClass
from schemas.object import Object
from schemas.place import Place
from schemas.agent import Agent
from schemas.relationship import Relationship

In [3]:
import pydantic
print(pydantic.__version__)

2.12.5


In [4]:
def get_batch(df: DataFrame, start: int, size: int):
    if start < 0:
        raise ValueError("start must be >= 0")

    if size < 0:
        raise ValueError("size must be >= 0")

    return df.iloc[start:start + size]

model = get_llm(0.7)

In [5]:
data_dir = "./data"
processed_dir = os.path.join(data_dir, "processed")
folktales_path = os.path.join(processed_dir, "folk_tales_deduplicated.csv")
folktales_df = load_csv(folktales_path)
selected_folktales_df = get_batch(folktales_df, 0, 5)

metadata_dir = "./metadata"
structure_dir = os.path.join(metadata_dir, "structure")
collections_dir = os.path.join(metadata_dir, "collections")
entities_dir = os.path.join(metadata_dir, "entities")

out_dir = "./out_prueba"
os.makedirs(out_dir, exist_ok=True)

structures = load_json_folder(structure_dir)
collections = load_json_folder(collections_dir)
entities = load_json_folder(entities_dir)

n_folktales = len(selected_folktales_df)

i = 0
row = selected_folktales_df.iloc[i]

In [8]:
load = True
text = row["text"]
url = row["source"]
nation = row["nation"]
title = row["title"]

filename = re.sub(clean_regex, "", title)
filename = title_case_to_snake_case(filename)

json_path = os.path.join(out_dir, f"{i}_{filename}_annotation.json")

if load and os.path.exists(json_path):

    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)
        
    genre = GenreClass(data["genre"])
    objects = [Object(**o) for o in data["objects"]]
    places = [Place(**p) for p in data["places"]]
    agents = [Agent(**a) for a in data["agents"]]
    relationships = [Relationship(**r) for r in data["relationships"]]
    story_segments = data.get("story_segments", [])


else:
    logger.debug(f"Starting annotation: '{title}' (idx={i})")

    genre = extract_genre(model, text, collections["genre"])
    objects = extract_objects(model, text, entities["object"])
    places = extract_places(model, text, entities["place"])
    agents = extract_agents(
        model, text, places,
        structures["role"],
        collections["trait"]
    )
    relationships = extract_relationships(model, text, agents)
    story_segments = extract_story_segments(model, text)

    # Serialización
    data = {
        "genre": genre.model_dump(),
        "objects": [o.model_dump() for o in objects],
        "places": [p.model_dump() for p in places],
        "agents": [a.model_dump() for a in agents],
        "relationships": [r.model_dump() for r in relationships],
        "story_segments": story_segments,
    }

    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)



In [7]:
events = []
for j, segment in tqdm(enumerate(story_segments), desc=f"Events ({title})", leave=False):
    objects_ids, place_id = extract_event_elements(model, title, segment, objects, places)

    event_type, final_thoughts = hierarchical_event_classification(
        model=model,
        event_index=j,
        story_segments=story_segments,
        taxonomy_tree=structures["function"],
        n_rounds=3,
        verbose=False
    )
    
    if event_type is None:
        event_type = EventClass.EVENT
    name = extract_event_name(model, event_type, segment, final_thoughts)
    event_agents = extract_event_agents(model, segment, final_thoughts, agents, title)
    
    # u = uuid.uuid4()
    # fixed = f"place_{u.hex}"

    event = Event(
        type=event_type,
        name=name,
        description=segment,
        agents=event_agents,
        objects=objects_ids,
        place=place_id,
        thoughts=final_thoughts
    )

    events.append(event)

folktale = Folktale(
    url=url,
    nation=nation,
    title=title,
    genre=genre,
    agents=agents,
    relationships=relationships,
    places=places,
    objects=objects,
    events=events
)

filename = re.sub(clean_regex, "", title)
filename = title_case_to_snake_case(filename)
path = os.path.join(out_dir, f"{i}_{filename}")
save_structured_folktale(folktale, path)



Events (Momotaro): 0it [00:00, ?it/s]2026-04-20 14:32:17.268 | DEBUG    | annotation.tools.event_extractor:extract_event_elements:99 - Attempt 1/5 for segment


System: You are an expert information extraction system specialized in narrative analysis.

Your task is to extract structured narrative elements from a folktale segment with high precision.

You will be given:
- The title of the folktale.
- A segment of the story.
- A list of candidate objects.
- A list of candidate locations.

EXTRACTION RULES:
   
1. Objects:
	- Select only objects that are explicitly mentioned or clearly used in the segment.
	- Do NOT include irrelevant or background objects.
	- If no objects are present, return an empty list.

3. Location:
	- Select the SINGLE most relevant location where the segment occurs.
	- If multiple locations are mentioned, choose the primary setting.
	- If the location is unclear, choose the best option from context.
			
STRICT CONSTRAINTS:						
- You MUST ONLY select elements from the provided lists.
- You MUST return indices (integers), NOT names.
- Each index can appear ONLY ONCE.
- Do NOT invent or assume elements not present in the l

2026-04-20 14:32:23.858 | DEBUG    | annotation.tools.event_extractor:extract_event_elements:122 - Valid extraction on attempt 1


System: You are an expert narrative annotation assistant.
Your task is to generate a concise for a narrative event.

INSTRUCTIONS:
- Produce a SHORT label (3-6 words).
- The name must capture the core action or outcome of the event.
- Prefer "verb + object" or "key event description" style.
- Do NOT repeat or paraphrase the full event text.
- Do NOT include unnecessary details.
- Do NOT include explanations.

OUTPUT FORMAT:
Return ONLY the event name as a single line.

Human: Event type:
setup

Event text:
If you’ll believe me there was a time when the fairies were none so shy as they are now. That was the time when beasts talked to men, when there were spells and enchantments and magic every day, when there was great store of hidden treasure to be dug up, and adventures for the asking.

Reasoning:
- The text describes a past time when magic and wonder existed, setting the stage for what's to come. It appears to be a nostalgic introduction to a story that may involve adventures, treasu

2026-04-20 14:32:40.957 | DEBUG    | annotation.tools.event_classifier:extract_event_name:416 - Event name: Past Time of Magic


...

System: You are an expert narrative annotation assistant.
Your task is to generate a concise for a narrative event.

INSTRUCTIONS:
- Produce a SHORT label (3-6 words).
- The name must capture the core action or outcome of the event.
- Prefer "verb + object" or "key event description" style.
- Do NOT repeat or paraphrase the full event text.
- Do NOT include unnecessary details.
- Do NOT include explanations.

OUTPUT FORMAT:
Return ONLY the event name as a single line.

Human: Event type:
resolution

Event text:
“Kia! Kia!” says the monkey. “Thanks, my Lord Momotaro.” “Ken! Ken!” says the pheasant. “Thanks, my Lord Momotaro.” “Bow! Wow! Wow!” says the dog. “Thanks, my dear Lord Momotaro.”

Reasoning:
- The story has reached its culmination with the successful retrieval of the Ogres' treasure and the defeat of the Ogres. This marks a turning point in the narrative as it transitions from the conflict to the resolution.
- The text describes the final confrontation between Momotaro and the 

2026-04-20 14:40:01.175 | DEBUG    | annotation.tools.event_classifier:extract_event_name:416 - Event name: Momotaro Defeats the Ogres


System: You are an expert information extraction system specialized in identifying which agents participate in a specific event within a folktale.

Your task is to determine which agents from a given candidate list are involved in the event described.

For each participating agent, you must extract:
- The agent's index in the provided list.
- The actions they perform in the event.
- Their importance in the event (0-10 scale)

IMPORTANCE SCALE:
- 0-3: minor or background presence.
- 4-6: relevant participation.
- 7-8: central participant.
- 9.10: critical to the event outcome.

INPUTS YOU WILL RECEIVE:
- Folktale title.
- Event text.
- List of candidate agents (with indices).
- Reasoning/thoughts from another model about the event.

IMPORTANT RULES:
- Only include agents that clearly participate in the event.
- Do NOT invent agents not present in the candidate list.
- The "id" must be the index of the agent in the provided list.
- Each agent must include at least one action.
- Actions m

2026-04-20 14:40:05.562 | DEBUG    | annotation.tools.event_classifier:extract_event_agents:515 - Agent=the monkey | actions=['joined the final confrontation', 'helped overcome the Ogres'] | importance=0.8
2026-04-20 14:40:05.562 | DEBUG    | annotation.tools.event_classifier:extract_event_agents:515 - Agent=the pheasant | actions=['joined the final confrontation', 'helped overcome the Ogres'] | importance=0.8
2026-04-20 14:40:05.562 | DEBUG    | annotation.tools.event_classifier:extract_event_agents:515 - Agent=the dog | actions=['joined the final confrontation', 'helped overcome the Ogres'] | importance=0.8
2026-04-20 14:40:05.563 | DEBUG    | annotation.tools.event_classifier:extract_event_agents:515 - Agent=the ogres | actions=['fought against Momotaro and his companions in the final confrontation', 'were defeated by them'] | importance=0.9


ValidationError: 1 validation error for Folktale
genre
  Input should be 'fable', 'fairy_tale', 'legend' or 'myth' [type=enum, input_value=Genre(genre=<GenreClass.FAIRY_TALE: 'fairy_tale'>), input_type=Genre]
    For further information visit https://errors.pydantic.dev/2.12/v/enum

In [10]:
folktale = Folktale(
    url=url,
    nation=nation,
    title=title,
    genre=genre,
    agents=agents,
    relationships=relationships,
    places=places,
    objects=objects,
    events=events
)

filename = re.sub(clean_regex, "", title)
filename = title_case_to_snake_case(filename)
path = os.path.join(out_dir, f"{i}_{filename}")
save_structured_folktale(folktale, path)

2026-04-20 14:43:49.101 | SUCCESS  | utils.loader:save_structured_folktale:56 - Annotated folktale saved sucessfully in ./out_prueba\0_momotaro
